In [1]:
import pandas as pd
import os
import numpy as np
import time
from skimage.transform import resize
from skimage.io import imread
from skimage.color import rgb2gray
from skimage.feature import graycomatrix, graycoprops
from sklearn import svm
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import kagglehub

In [2]:
# construct classes array
path = kagglehub.dataset_download("ossossh/solar-pannels-image-enhanced")

classes_path = os.path.join(path, 'Data4T2')
classes = [d for d in os.listdir(classes_path) if os.path.isdir(os.path.join(classes_path, d))]

print(classes)

Using Colab cache for faster access to the 'solar-pannels-image-enhanced' dataset.
['Snow-Covered', 'Dusty', 'Electrical-damage', 'Clean', 'Bird-drop', 'Physical-Damage']


In [3]:
# GLCM Texture Feature Extraction
def extract_glcm_features(image, distances=[1], angles=[0, np.pi/4, np.pi/2, 3*np.pi/4]):
    # Rescale image to 0-255 and convert to uint8 (required by graycomatrix)
    rescaled = (image * 255).astype(np.uint8)

    # Calculate GLCM
    glcm = graycomatrix(rescaled, distances=distances, angles=angles,
                        levels=256, symmetric=True, normed=True)

    # Calculate properties from GLCM
    contrast = graycoprops(glcm, 'contrast').flatten()
    dissimilarity = graycoprops(glcm, 'dissimilarity').flatten()
    homogeneity = graycoprops(glcm, 'homogeneity').flatten()
    energy = graycoprops(glcm, 'energy').flatten()
    correlation = graycoprops(glcm, 'correlation').flatten()
    ASM = graycoprops(glcm, 'ASM').flatten()

    # Combine all features into a single array
    features = np.hstack([contrast, dissimilarity, homogeneity, energy, correlation, ASM])

    return features

In [4]:
# Extract GLCM features from our dataset

flat_data_arr = []
target_arr = []

# Define valid image extensions
valid_extensions = ('.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.tif', '.JPG', '.JPEG', '.PNG', '.BMP', '.TIFF', '.TIF')

for category in classes:
    print(f'Processing category: {category}')
    class_path = os.path.join(classes_path, category)

    for root, dirs, files in os.walk(class_path):
        for img in files:
            if not img.lower().endswith(valid_extensions):
                continue
            try:
                img_full_path = os.path.join(root, img)
                img_array = imread(img_full_path)
                img_resized = resize(img_array, (256, 256, 3))
                gray_image = rgb2gray(img_resized)

                # Extract GLCM features
                glcm_features = extract_glcm_features(gray_image)

                # Add features to our dataset
                flat_data_arr.append(glcm_features)
                target_arr.append(classes.index(category))

            except Exception as e:
                print(f"Error processing {img}: {e}")
                continue


# Convert to numpy arrays
glcm_features = np.array(flat_data_arr)
glcm_targets = np.array(target_arr)

print("GLCM feature extraction completed.")
print(f"Feature shape: {glcm_features.shape}")
print(f"Number of features per image: {glcm_features.shape[1]}")

Processing category: Snow-Covered
Processing category: Dusty
Processing category: Electrical-damage
Processing category: Clean
Processing category: Bird-drop
Processing category: Physical-Damage
GLCM feature extraction completed.
Feature shape: (5638, 24)
Number of features per image: 24


In [5]:
# Create a DataFrame with the GLCM features
glcm_df = pd.DataFrame(glcm_features)
glcm_df['Target'] = glcm_targets

# Split the data
X_glcm = glcm_df.iloc[:, :-1]
y_glcm = glcm_df.iloc[:, -1]

# Split into training and testing sets
X_train_glcm, X_test_glcm, y_train_glcm, y_test_glcm = train_test_split(
    X_glcm, y_glcm, test_size=0.20, random_state=77, stratify=y_glcm
)

In [6]:
# Train SVM model with GLCM features

# Define SVM parameters
svm_params = {
    'C': 100,
    'gamma': 'scale',
    'kernel': 'rbf',
    'probability': True,
    'random_state': 42
}

# Create pipeline for GLCM features (scaling and SVM)
glcm_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', svm.SVC(**svm_params))
])

print("Training SVM with GLCM features...")
start_time = time.time()

# Train the model
glcm_pipeline.fit(X_train_glcm, y_train_glcm)

end_time = time.time()
training_time = end_time - start_time
print(f"Training completed in {training_time:.2f} seconds")

# Make predictions on test set
y_pred_glcm = glcm_pipeline.predict(X_test_glcm)


accuracy_glcm = accuracy_score(y_test_glcm, y_pred_glcm)
print(f"Test accuracy with GLCM+SVM: {accuracy_glcm:.4f}")

Training SVM with GLCM features...
Training completed in 10.63 seconds
Test accuracy with GLCM+SVM: 0.6809
